# Estructuración de un modelo de aprendizaje profundo
**Propósito**: Estructurar un proyecto de aprendizaje profundo considerando métricas, validación y optimización.

En esta actividad pondrás en práctica los conocimientos adquiridos sobre el proceso de diseño, validación y optimización de modelos de aprendizaje profundo. El objetivo es que desarrolles la capacidad de estructurar un modelo de manera sistemática, aplicando criterios de evaluación rigurosos y estrategias de mejora continua.

En el ejercicio deberás analizar un conjunto de datos, construir un modelo base, ajustarlo e interpretar sus resultados para garantizar que su rendimiento sea sólido y generalizable.

El proyecto se basa en el Adult Income Dataset, cuyo propósito es predecir si una persona gana más de $50,000 USD al año a partir de variables como su nivel educativo, ocupación, estado civil y horas de trabajo por semana. Este conjunto de datos te permitirá enfrentarte a desafíos comunes en la práctica profesional, como el manejo de datos desbalanceados, la selección de métricas adecuadas y la evaluación de la capacidad de generalización de un modelo.

#### Detalles del conjunto de datos
El dataset contiene las siguientes variables:

| Variable | Tipo | Descripción |
|-----------|------|--------------|
| **age** | Numérica | Edad del individuo (en años). |
| **workclass** | Categórica | Tipo de empleo o sector laboral (por ejemplo: Private, Self-emp, Government, etc.). |
| **fnlwgt** | Numérica | Peso muestral asignado por el censo (representa cuántas personas similares existen en la población). |
| **education** | Categórica | Nivel educativo alcanzado (por ejemplo: Bachelors, HS-grad, Masters, etc.). |
| **education-num** | Numérica | Nivel educativo expresado en número (por ejemplo: 9 = HS-grad, 13 = Bachelors). |
| **marital-status** | Categórica | Estado civil (por ejemplo: Married-civ-spouse, Never-married, Divorced). |
| **occupation** | Categórica | Tipo de ocupación (por ejemplo: Tech-support, Craft-repair, Sales, etc.). |
| **relationship** | Categórica | Relación familiar dentro del hogar (por ejemplo: Husband, Wife, Own-child). |
| **race** | Categórica | Raza o grupo étnico declarado (por ejemplo: White, Black, Asian-Pac-Islander, etc.). |
| **sex** | Categórica | Sexo biológico (Male o Female). |
| **capital-gain** | Numérica | Ganancia de capital (por ejemplo, ingresos por inversiones). |
| **capital-loss** | Numérica | Pérdida de capital (por ejemplo, pérdidas por inversiones). |
| **hours-per-week** | Numérica | Número promedio de horas trabajadas por semana. |
| **native-country** | Categórica | País de origen o nacionalidad. |
| **income** | Categórica | Etiqueta objetivo: indica si el ingreso anual es `<=50K` o `>50K`. |

Existen 48,842 filas de datos pero 3620 contienen datos faltantes. Al final resultan 45,222 muestras completas.
El dataset que se te ha proporcionado corresponde a la versión original disponible en UCI. Este dataset viene ya particionado en dos conjuntos: adult.data y adult.test. Sin embargo, para esta actividad concatenaremos dichas particiones para aprovechar ambos conjuntos y te daremos la oportunidad de que realices las particiones según tus criterios.

#### Estructura de la libreta.
La libreta está organizada en las siguientes secciones:

1. Exploración de datos, definición de métricas de rendimiento y partición de datos.
2. Diseño de un modelo de referencia (baseline)
3. Entrenamiento del modelo base
4. Evaluación e interpretación del rendimiento
5. Ajuste de hiperparámetros
6. Pruebas de generalización

#### Importar librerias necesarias

In [ ]:
import os
import math
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    precision_recall_curve, roc_curve, log_loss
)
from sklearn.inspection import permutation_importance

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

RANDOM_STATE = 42
DATA_PATH = "data/adult_UCI/adult.data"
TEST_PATH = "data/adult_UCI/adult.test"
TARGET_COL = "income"
POSITIVE_LABEL = ">50K"

pd.set_option('display.max_columns', 200)
np.random.seed(RANDOM_STATE)


## 1. Exploración de datos, definición de métricas de rendimiento y partición de datos

#### 1.1 Carga de datos y limpieza

In [ ]:
def cargar_datos(path, skiprows=0):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"No se encontró {path}. Coloca el archivo 'adult.csv' en este directorio."
        )

    column_names = [
        "age",
        "workclass",
        "fnlwgt",
        "education",
        "education-num",
        "marital-status",
        "occupation",
        "relationship",
        "race",
        "sex",
        "capital-gain",
        "capital-loss",
        "hours-per-week",
        "native-country",
        "income"
    ]

    df = pd.read_csv(path, na_values='?', skipinitialspace=True, header=None, names=column_names, skiprows=skiprows)

    for c in df.select_dtypes(include="object"):
        df[c] = df[c].str.strip()

    print("Datos en el dataset: ", df.shape[0])
        
    return df

In [ ]:
df_data = cargar_datos(DATA_PATH)
df_test = cargar_datos(TEST_PATH, skiprows=1)         # Este archivo tiene una fila extra al inicio
df = pd.concat([df_data, df_test], ignore_index=True)

In [ ]:
df_data.head()

In [ ]:
df_test.head()

In [ ]:
# En el conjunto de prueba, la columna objetivo tiene un punto al final de cada valor. La corregimos:
df_test["income"] = df_test["income"].str.replace('.', '', regex=False).str.strip()
df_test["income"].unique()


In [ ]:
# concatenamos los dataframes
df = pd.concat([df_data, df_test], ignore_index=True)
df.head()
print(len(df_data), len(df_test), len(df))

In [ ]:
def porcentaje_valores_faltantes(df):
    faltantes = df.isna().sum()
    porcentaje_faltantes = (faltantes / len(df)) * 100

    faltantes_table = pd.DataFrame({
        'Valores faltantes': faltantes,
        '% del total': porcentaje_faltantes
    }).sort_values(by='Valores faltantes', ascending=False)

    print(faltantes_table)

print("\nTotal de valores faltantes: ")
porcentaje_valores_faltantes(df)

# En caso que necesites revisar valores faltantes por conjunto de datos, descomenta las siguientes líneas:
#print("Valores faltantes en data: ")
#calcular_valores_faltantes(df_data)
#print("\n Valores faltantes en test")
#calcular_valores_faltantes(df_test)


Preguntas de reflexión:

1. ¿Conviene eliminar las filas o imputal valores?
2. ¿Qué impacto puede tener esto en la generalización del modelo?
3. Explica aquí que has decidido hacer con los datos faltantes y ¿Por qué?

In [ ]:
# Realiza en esta celda la imputación de valores faltantes.

In [ ]:
# Mostrar duplicados

duplicados = df[df.duplicated(keep=False)].copy()
duplicados = duplicados.sort_values(by=df.columns.tolist()).reset_index(drop=True)
duplicados.head(4)


Reflexión:

Como puedes observar en la celda anterior, el dataset tiene algunas filas duplicadas. 

1. ¿Significa esto que son errores de captura y que debemos eliminar las filas? Justifica tu respuesta

In [ ]:
# Si decides eliminar duplicados, hazlo en esta celda.

#### 1.2 Análisis de datos exploratorio

In [ ]:
# revisamos información acerca del dataframe resultante
df.info()

Instrucciones:

Crea código para visualizar histogramas para todas las variables númericas.

In [ ]:
# Muestra histogramas para todas las variables numéricas

**Edita la celda para dar respuesta a las siguientes preguntas de reflexión:**

Reflexión:
- ¿A qué conclusión llegas observando las visualizaciones? 
- ¿Existen valores atípicos?
- ¿Realizarás algún procedimiento extra en los datos después de visualizar las gráficas?

Nota: 
Si decides hacer algún procesamiento extra a las variables númericas en este punto, agrega celdas abajo.

In [ ]:
# Crea visualizaciones para las variables categóricas aquí
# Piensa en qué tipo de gráficos serían más útiles para visualizar potenciales correlaciones entre la variable a predecir y las variables predictoras. 

Reflexión:
- ¿Qué variables consideras que tendrán un mayor impacto en la predicción?

#### Partición de los datos

In [ ]:
# Primeramente revisa la distribución de clases, para determinar si es necesario aplicar técnicas para balancear las clases. 

counts = df["income"].value_counts()
print(counts)

positive_class_percentage = counts.iloc[1] / counts.sum() * 100
print(f'Porcentaje de la clase positiva (1): {positive_class_percentage:.2f}%')
negative_class_percentage = counts.iloc[0] / counts.sum() * 100
print(f'Porcentaje de la clase negativa (0): {negative_class_percentage:.2f}%')

Reflexión:
- ¿Qué grado de desbalance tienen las clases?
- ¿Estás considerando implementar una técnica de balanceo?

In [ ]:
# Completa el código en esta celda para crear particiones para tus datos.

X = df.drop(columns=[TARGET_COL])                   # Removemos completamente la variable a predecir del dataframe
y = (df[TARGET_COL] == POSITIVE_LABEL).astype(int)  # 1 si >50K, 0 en otro caso


# Coloca tu código aqui para formar tus conjuntos de entrenamiento
# La idea es que crees tres conjuntos de datos:
# X_train , y_train
# X_val   , y_val
# X_test  , y_test 

# Reemplaza las siguientes líneas con tu código, asegurándote de crear los conjuntos de datos indicados arriba:
X_temp, X_test, y_temp, y_test = None, None, None, None
X_train, X_val, y_train, y_val = None, None, None, None

# Imprimimos la distribución de clases

"""
print("Distribución de clases:")
for name, yy in {"train": y_train, "valid": y_val, "test": y_test}.items():
    pos_rate = yy.mean()
    print(f"{name:>5s}: n={len(yy):4d}, %pos={pos_rate*100:5.2f}")
"""

Reflexión:

- ¿En qué casos es importante dividir en tres conjuntos de datos?
- ¿Qué papel juega cada uno de los conjuntos?
- ¿Qué porcentaje de datos le asignaron a cada conjunto?
- ¿Utilizaron muestreo estratificado? ¿Por qué? ¿Qué podría suceder si no se usa muestreo estratificado?


#### Preprocesamiento

Instrucciones:

Las variables categóricas tienen que ser codificadas adecuadamente para que puedan ser aprovechadas por el modelo. Completa el código abajo para codificar las variables utilizando One Hot Encodings.

In [ ]:

columnas_numericas = X.select_dtypes(include=[np.number]).columns.tolist()
columnas_categoricas = X.select_dtypes(exclude=[np.number]).columns.tolist()

# Creamos una tubería de procesamiento para las variables numéricas -----------------------------------

num_pipe = Pipeline([
    # El imputador no modifica nada si no hay valores faltantes.
    # Se mantiene para mostrar una práctica general de robustez.
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Creamos una tubería de procesamiento para las variables categóricas -----------------------------------
# Probar bucket de infrecuentes si está disponible en la versión de sklearn
try:
    # Agrupa las categorias infrecuentes (<min_frequency) en una sola categoría
    # (requiere sklearn >= 1.1)
    
    ohe = OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=50,
        drop=None
    )
    print("Usando 'infrequent_if_exist' para OneHotEncoder")
except TypeError:
    # Caso seguro
    ohe = OneHotEncoder(handle_unknown="ignore", drop=None)
    print("Usando 'ignore' para OneHotEncoder")

cat_pipe = Pipeline([
    ("ohe", ohe)
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, columnas_numericas),
    ("cat", cat_pipe, columnas_categoricas)
])


Reflexiona:

- ¿Por qué es importante incluir StandarScaler() en la tuberia de procesamiento de las variables numéricas?
- ¿Cuál es la diferencia entre utilizar la opción "infrequent_if_exist" e "ignore" para el manejo de categorias desconocidas?
- ¿Por qué en la tuberia para variables categoricas ya no es necesario incluir el StandardScaler?.

#### Definición de métricas

Reflexiona:

- ¿Qué tipo de error consideras más costoso para este problema?
- ¿Qué métrica principal planeas usar? ¿Por qué?

In [ ]:
# Remplaza None con la métrica que consideres más adecuada para evaluar el desempeño de tu modelo.
# El código en las siguientes celdas depende de la definición de esta variable.
# Si decides utilizar otra métrica, es posible que necesites ajustar las funciones de evaluación utilizadas posteriormente.

PRIMARY_METRIC = None  # opciones: {"precision", "recall", "f1", "roc_auc", "average_precision"}

In [ ]:
# Esta celda define funciones para evaluar modelos y barrer umbrales

def summarize_threshold_metrics(y_trainue, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)
    out = {
        "accuracy": accuracy_score(y_trainue, y_pred),
        "precision": precision_score(y_trainue, y_pred, zero_division=0),
        "recall": recall_score(y_trainue, y_pred, zero_division=0),
        "f1": f1_score(y_trainue, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_trainue, y_proba),
        "average_precision": average_precision_score(y_trainue, y_proba),
        "threshold": threshold
    }
    return out

def sweep_threshold_by_metric(y_trainue, y_proba, metric="f1"):
    # Barrido con puntos de la curva PR para eficiencia
    prec, rec, thr = precision_recall_curve(y_trainue, y_proba)
    # En precision_recall_curve, 'thr' tiene len = len(prec)-1
    best = {"metric": -1, "threshold": 0.5}
    for t in np.unique(np.concatenate([thr, [0.5]])):
        y_pred = (y_proba >= t).astype(int)
        if metric == "f1":
            val = f1_score(y_trainue, y_pred, zero_division=0)
        elif metric == "average_precision":
            # AP no depende de umbral; mantener como referencia
            val = average_precision_score(y_trainue, y_proba)
        else:
            raise ValueError("Métrica no soportada para sweep")
        if val > best["metric"]:
            best = {"metric": val, "threshold": t}
    return best

def evaluate_pipeline(model, X_train, y_train, X_val, y_val, primary_metric="f1", do_plots=False, label="modelo"):
    # Ajuste y evaluación en validación
    model.fit(X_train, y_train)
    val_proba = model.predict_proba(X_val)[:, 1]
    best = sweep_threshold_by_metric(y_val, val_proba, metric=primary_metric)
    metrics = summarize_threshold_metrics(y_val, val_proba, threshold=best["threshold"])
    metrics["primary_metric_value"] = best["metric"]
    metrics["label"] = label

    if do_plots:
        # PR Curve
        prec, rec, _ = precision_recall_curve(y_val, val_proba)
        plt.figure()
        plt.plot(rec, prec)
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title(f"Precision-Recall — {label}")
        plt.show()

        # ROC Curve
        fpr, tpr, _ = roc_curve(y_val, val_proba)
        plt.figure()
        plt.plot(fpr, tpr)
        plt.xlabel("FPR")
        plt.ylabel("TPR")
        plt.title(f"ROC — {label}")
        plt.show()

        # Confusion matrix at chosen threshold
        y_pred = (val_proba >= metrics["threshold"]).astype(int)
        cm = confusion_matrix(y_val, y_pred)
        plt.figure()
        plt.imshow(cm, cmap=None)
        plt.title(f"Matriz de confusión (val) — {label}")
        plt.xlabel("Predicho")
        plt.ylabel("Real")
        for (i,j),v in np.ndenumerate(cm):
            plt.text(j, i, str(v), ha='center', va='center')
        plt.show()

    return metrics, model, best["threshold"]

def evaluate_model(model, X_val, y_val, primary_metric="f1", do_plots=False, label="modelo"):
    # Ajuste y evaluación en validación
    val_proba = model.predict_proba(X_val)[:, 1]
    best = sweep_threshold_by_metric(y_val, val_proba, metric=primary_metric)
    metrics = summarize_threshold_metrics(y_val, val_proba, threshold=best["threshold"])
    metrics["primary_metric_value"] = best["metric"]
    metrics["label"] = label

    if do_plots:
        # PR Curve
        prec, rec, _ = precision_recall_curve(y_val, val_proba)
        plt.figure()
        plt.plot(rec, prec)
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title(f"Precision-Recall — {label}")
        plt.show()

        # ROC Curve
        fpr, tpr, _ = roc_curve(y_val, val_proba)
        plt.figure()
        plt.plot(fpr, tpr)
        plt.xlabel("FPR")
        plt.ylabel("TPR")
        plt.title(f"ROC — {label}")
        plt.show()

        # Confusion matrix at chosen threshold
        y_pred = (val_proba >= metrics["threshold"]).astype(int)
        cm = confusion_matrix(y_val, y_pred)
        plt.figure()
        plt.imshow(cm, cmap=None)
        plt.title(f"Matriz de confusión (val) — {label}")
        plt.xlabel("Predicho")
        plt.ylabel("Real")
        for (i,j),v in np.ndenumerate(cm):
            plt.text(j, i, str(v), ha='center', va='center')
        plt.show()

    return metrics, model, best["threshold"]

## 2. Diseño de un modelo de referencia. 
Definimos dos modelos de referencia: Un clasificador de clase mayoritaria y regresión logística

In [ ]:
# En esta celda definimos los modelos de referencia y los evaluamos.

baseline_dummy = Pipeline([
    ("prep", preprocessor),
    ("clf", DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE))
])

baseline_logit = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, n_jobs=None, random_state=RANDOM_STATE))
])

metrics_dummy, model_dummy, thr_dummy = evaluate_pipeline(
    baseline_dummy, X_train, y_train, X_val, y_val, primary_metric=PRIMARY_METRIC, do_plots=False, label="Dummy (mayoritaria)"
)
metrics_reglog, model_log, thr_log = evaluate_pipeline(
    baseline_logit, X_train, y_train, X_val, y_val, primary_metric=PRIMARY_METRIC, do_plots=True, label="Regresión Logística"
)

pd.DataFrame([metrics_dummy, metrics_reglog])[["label","primary_metric_value","accuracy","precision","recall","f1","roc_auc","average_precision","threshold"]]


Reflexión:

- ¿Cómo fue el desempeño de los modelos de referencia?

## 3. Entrenamiento de un modelo base

In [ ]:
# Definimos el pipeline base para el MLP
mlp_base_pipe = Pipeline([
    ("prep", preprocessor),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        alpha=1e-4,
        learning_rate_init=1e-3,
        early_stopping=True,
        random_state=RANDOM_STATE,
        max_iter=1,
        warm_start=True  # Permite entrenamiento iterativo
    ))
])


# Aplicamos el preprocesamiento a los datos
preprocessor = mlp_base_pipe.named_steps["prep"]
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

mlp_base = mlp_base_pipe.named_steps["clf"]

# Realizamos el entrenamiento
# Usamos entrenamiento manual en lugar de evaluate_pipeline() para guardar métricas por época

n_epochs = 20
train_acc, val_acc = [], []
train_loss, val_loss = [], []

best_val_loss = np.inf
best_epoch = 0
best_model = None

for epoch in range(n_epochs):
    mlp_base.fit(X_train_processed, y_train)
    y_tr_proba = mlp_base.predict_proba(X_train_processed)[:,-1]
    y_val_proba = mlp_base.predict_proba(X_val_processed)[:,-1]
    y_tr_pred = mlp_base.predict(X_train_processed)
    y_val_pred = mlp_base.predict(X_val_processed)

    train_acc.append(accuracy_score(y_train, y_tr_pred))
    val_acc.append(accuracy_score(y_val, y_val_pred))
    tr_loss = log_loss(y_train, y_tr_proba)
    vl_loss = log_loss(y_val, y_val_proba)

    train_loss.append(tr_loss)
    val_loss.append(vl_loss)
    print(f"Época {epoch+1:2d}: Train Acc={train_acc[-1]:.4f}, Val Acc={val_acc[-1]:.4f}, Train Loss={tr_loss:.4f}, Val Loss={vl_loss:.4f}")

    # Guardamos el mejor modelo
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        best_epoch = epoch
        best_mlp_base = copy.deepcopy(mlp_base)

# Generamos gráficas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- (a) Curva de pérdida
axes[0].plot(train_loss, label="Pérdida (train)", color="tab:blue")
axes[0].plot(val_loss, label="Pérdida (val)", color="tab:orange")
axes[0].set_title("Evolución de la pérdida")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Log-Loss")
axes[0].legend()
axes[0].grid(True)
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# --- (b) Curva de accuracy
axes[1].plot(train_acc, label="Accuracy (train)", color="tab:blue")
axes[1].plot(val_acc, label="Accuracy (val)", color="tab:orange")
axes[1].set_title("Evolución del accuracy")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True)
axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.suptitle("Curvas de entrenamiento — Loss y Accuracy (Train vs Validación)", fontsize=14)
plt.tight_layout()
plt.show()
print(f"Mejor validación en época {np.argmax(val_acc)} con accuracy={max(val_acc):.3f}")

Reflexion:

- De acuerdo a la figura anterior ¿En que época consideras que se comienza a sobreajustar del modelo?


# 4. Evaluación e interpretación del rendimiento

In [ ]:
# Evaluamos el mejor modelo en el conjunto de validación

metrics_mlp_base, model_mlp_base, thr_mlp_base = evaluate_model(best_mlp_base, X_val_processed, y_val, primary_metric=PRIMARY_METRIC, do_plots=True, label="MLP (mejor modelo)")

In [ ]:
pd.DataFrame([metrics_dummy, metrics_reglog, metrics_mlp_base])[["label","primary_metric_value","accuracy","precision","recall","f1","roc_auc","average_precision","threshold"]]


Reflexión:

- ¿Cómo fue el desempeño del modelo base con respecto a los baselines?

In [ ]:
# Esta celda calcula y muestra las brechas entre entrenamiento y validación

from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, accuracy_score

def check_set_performance_gaps(model, X_trainain, y_trainain, X_val, y_val, threshold=0.5):
    p_train = model.predict_proba(X_trainain)[:,1]
    p_val = model.predict_proba(X_val)[:,1]
    
    y_pred_train = (p_train >= threshold).astype(int)
    y_pred_val = (p_val >= threshold).astype(int)
    
    metrics = {
        "f1_train": f1_score(y_trainain, y_pred_train),
        "f1_val": f1_score(y_val, y_pred_val),
        "roc_train": roc_auc_score(y_trainain, p_train),
        "roc_val": roc_auc_score(y_val, p_val),
        "ap_train": average_precision_score(y_trainain, p_train),
        "ap_val": average_precision_score(y_val, p_val),
        "acc_train": accuracy_score(y_trainain, y_pred_train),
        "acc_val": accuracy_score(y_val, y_pred_val)
    }
    df = pd.DataFrame([metrics])
    display(df)

    print("Brechas (train - val):")
    for k in ["f1","roc","ap","acc"]:
        gap = metrics[f"{k}_train"] - metrics[f"{k}_val"]
        print(f"{k.upper():<5} gap: {gap:.3f}")

check_set_performance_gaps(best_mlp_base, X_train_processed, y_train, X_val_processed, y_val)


Reflexión:
- De acuerdo a las brechas en el desempeño en entrenamiento y validación:
- ¿Qué nivel de ajuste crees que tenga el modelo (sobreajustado, subajustado o un buen ajuste)? ¿Por qué? 

## 5. Ajuste de hiperparámetros

In [ ]:
# Vamos a realizar una búsqueda aleatoria de hiperparámetros para el MLP. 
# Dependiendo de tu hardware, puedes ajustar el número de combinaciones a evaluar. 

from sklearn.model_selection import ParameterSampler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np

rng = np.random.default_rng(RANDOM_STATE)

# Espacio de búsqueda reducido
space = {
    "hidden_layer_sizes": [(32,), (64,), (64,32)],
    "alpha": [1e-4, 1e-3, 1e-2],
    "learning_rate_init": [5e-4, 1e-3],
    "batch_size": [64, 128]
}

# Genera 6 combinaciones aleatorias
candidates = list(ParameterSampler(space, n_iter=6, random_state=RANDOM_STATE))

results = []
best_tuned_model = None
best_tuned_threshold = 0.5
best_tuned_score = -np.inf

for params in candidates:
    mlp = Pipeline([
        ("prep", preprocessor),
        ("clf", MLPClassifier(
            hidden_layer_sizes=params["hidden_layer_sizes"],
            activation="relu",
            alpha=params["alpha"],
            learning_rate_init=params["learning_rate_init"],
            early_stopping=True,
            batch_size=params["batch_size"],
            random_state=RANDOM_STATE,
            max_iter=200
        ))
    ])
    metrics, fitted, thr = evaluate_pipeline(
        mlp, X_train, y_train, X_val, y_val,
        primary_metric=PRIMARY_METRIC, do_plots=False,
        label=f"MLP {params['hidden_layer_sizes']}, a={params['alpha']}, lr={params['learning_rate_init']}, bs={params['batch_size']}"
    )
    row = {
        "hls": params["hidden_layer_sizes"],
        "alpha": params["alpha"],
        "lr": params["learning_rate_init"],
        "batch": params["batch_size"],
        "primary": metrics["primary_metric_value"],
        "f1": metrics["f1"],
        "ap": metrics["average_precision"]
    }
    results.append(row)
    score = metrics["primary_metric_value"]
    if score > best_tuned_score:
        best_tuned_score = score
        best_tuned_model = fitted
        best_tuned_threshold = thr

df_tuning = pd.DataFrame(results).sort_values(by="primary", ascending=False)
df_tuning.head(10)


Reflexión:
- ¿Qué combinación de hiperparámetros provee los mejores resultados?
- ¿Existe una mejora significativa?
- ¿Crees que una busqueda de parámetros por rejilla obtenga mejores resultados?
- ¿Cuál es la diferencia entre usar el método aleatorio y el de la rejilla?

In [ ]:
# Intenta aquí una busqueda en rejilla para el MLP. Recuerda aprovechar el parámetro jobs para especificar el número de trabajos en paralelo y reducir el tiempo de búsqueda.

In [ ]:
# Re-evaluar el mejor modelo ajustado después de la búsqueda de hiperparámetros en validación para resumen

def evaluate_label(estimator, label):
    proba = estimator.predict_proba(X_val)[:,1]
    best = sweep_threshold_by_metric(y_val, proba, metric=PRIMARY_METRIC)
    summ = summarize_threshold_metrics(y_val, proba, threshold=best["threshold"])
    summ["primary_metric_value"] = best["metric"]
    summ["label"] = label
    return summ, best["threshold"]

metrics_tuned_val, thr_final_val = evaluate_label(best_tuned_model, "MLP Ajustado")
summary_val = pd.DataFrame([metrics_dummy, metrics_reglog, metrics_mlp_base, metrics_tuned_val])[
    ["label","primary_metric_value","accuracy","precision","recall","f1","roc_auc","average_precision","threshold"]
]
summary_val


## 6. Pruebas de generalización: 
Evaluación en test (umbral congelado del mejor modelo)

In [ ]:
final_model = best_tuned_model
final_threshold = best_tuned_threshold
final_score = best_tuned_score

In [ ]:

def test_report(estimator, threshold, label):
    proba = estimator.predict_proba(X_test)[:,1]
    summ = summarize_threshold_metrics(y_test, proba, threshold=threshold)
    summ["label"] = label
    return summ

rep_dummy = test_report(model_dummy, thr_dummy, "Dummy (clase mayoritaria)")
rep_log = test_report(model_log, thr_log, "Regresion Logística")
rep_mlp_base = test_report(mlp_base_pipe, thr_mlp_base, "MLP Base")
rep_mlp_ajustado = test_report(best_tuned_model, best_tuned_threshold, "MLP Ajustado")
rep_mlp_final = test_report(final_model, final_threshold, "MLP Final")

summary_testst = pd.DataFrame([rep_dummy, rep_log, rep_mlp_base, rep_mlp_ajustado, rep_mlp_final])[
    ["label","accuracy","precision","recall","f1","roc_auc","average_precision","threshold"]
]
summary_testst


Reflexión:
- ¿Qué modelo generaliza mejor?
- ¿Cuál modelo elegirías para producción?

In [ ]:
# Análisis de métricas por subgrupo en el conjunto de prueba para revision de equidad

def subgroup_metrics(estimator, X, y, column, threshold, min_count=30, require_both_classes=True):
    if column not in X.columns:
        return None

    res = []
    col_vals = X[column].astype(str)
    for val in col_vals.unique():
        idx = (col_vals == val)
        n = int(idx.sum())
        y_sub = y.loc[idx]
        pos = int(y_sub.sum())
        neg = int(n - pos)

        row = {"grupo": f"{column}={val}", "n": n, "pos": pos, "neg": neg}

        # Reglas de soporte
        if n < min_count or (require_both_classes and y_sub.nunique() < 2):
            # Métricas no evaluables con poco soporte o 1 sola clase
            row.update({m: np.nan for m in
                        ["accuracy","precision","recall","f1","roc_auc","average_precision"]})
            row["nota"] = "soporte insuficiente o 1 sola clase"
        else:
            proba = estimator.predict_proba(X.loc[idx])[:, 1]
            summ = summarize_threshold_metrics(y_sub, proba, threshold=threshold)
            row.update({k: summ[k] for k in
                        ["accuracy","precision","recall","f1","roc_auc","average_precision"]})
            row["nota"] = ""

        res.append(row)

    cols = ["grupo","n","pos","neg","accuracy","precision","recall","f1","roc_auc","average_precision","nota"]
    return (pd.DataFrame(res)
              .sort_values(by=["nota","f1"], ascending=[True, False])
              .reset_index(drop=True))


cols_sub = [c for c in ["sex","race","education"] if c in X.columns]

sub_tables = {}
for col in cols_sub:
    sub_tables[col] = subgroup_metrics(final_model, X_test, y_test, col, final_threshold)

for col, tbl in sub_tables.items():
    print(f"\nMétrica por subgrupo — {col}")
    display(tbl)


Reflexión:
 - ¿Qué observas en las métricas por subgrupo? 
 - ¿Hay algún subgrupo con desempeño significativamente peor?

## Resumen final

In [ ]:

print("Validación (resumen):")
display(summary_val)

print("\nTest (resumen):")
display(summary_testst)

print("Mejor umbral (validación) para el modelo de regresión logística:", thr_log)
print("Mejor umbral (validación) para el modelo base MLP:", thr_mlp_base)
print("Mejor umbral (validación) para el modelo sintonizado:", final_threshold)


In [ ]:

# Ejecuta
print("-"*10 + "Brechas en validación" + "-"*10)
check_set_performance_gaps(final_model, X_train, y_train, X_val, y_val)

print("\n")
print("-"*10 + "Brechas en test" + "-"*10)
check_set_performance_gaps(final_model, X_train, y_train, X_test, y_test)


Reflexión final:
- ¿Cuál modelo elegirías para producción y por qué (balance entre precisión, recall, F1, AUC, estabilidad)?
- ¿Qué otras estrategias aplicarías para mejorar el modelo?
- ¿Qué plan de monitoreo y alertas propones para detectar deriva del modelo y degradación de métricas en producción?
- ¿Con qué frecuencia y con qué criterio reentrenarías el modelo? ¿Cómo asegurarías la calidad de nuevas etiquetas?
- ¿Qué experimentos prioritarios faltan (más búsqueda de hiperparámetros, ensambles, calibración, ajuste de clases, arquitecturas alternativas)?

